In [4]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Ananya Manna\OneDrive\Desktop\FINAL PROJECT\Final_Project\Dataset\final dataset.csv")
df.head()

,DATASET,SERIES_CODE,OBS_MEASURE,COUNTRY,INDICATOR,FREQUENCY,SCALE,1995,1996,1997,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,IMF.RES:WEO(9.0.0),GBR.BCA_NGDPD.A,OBS_VALUE,United Kingdom,"Current account balance (credit less debit), P...",Annual,Units,-0.381,-0.264,-0.140,...,-5.022,-4.948,-5.448,-3.493,-3.927,-2.688,-2.934,-0.437,-2.102,-3.509
1,IMF.RES:WEO(9.0.0),USA.GGX_NGDP.A,OBS_VALUE,United States,"Expenditure, General government, Percent of GDP",Annual,Units,NaN,NaN,NaN,...,35.324,35.031,35.333,35.194,35.349,35.819,44.722,43.201,36.790,37.650
2,IMF.RES:WEO(9.0.0),AUT.BCA_NGDPD.A,OBS_VALUE,Austria,"Current account balance (credit less debit), P...",Annual,Units,-2.876,-2.854,-2.559,...,2.397,1.577,2.574,1.264,0.846,2.376,3.367,1.735,-0.862,1.341
3,IMF.RES:WEO(9.0.0),USA.GGXWDG_NGDP.A,OBS_VALUE,United States,"Gross debt, General government, Percent of GDP",Annual,Units,NaN,NaN,NaN,...,104.930,105.424,107.380,106.374,107.625,108.751,132.513,125.009,119.104,119.836
4,IMF.RES:WEO(9.0.0),GBR.GGX_NGDP.A,OBS_VALUE,United Kingdom,"Expenditure, General government, Percent of GDP",Annual,Units,37.631,35.716,34.891,...,41.184,40.425,39.632,39.155,38.871,38.763,49.987,45.753,44.099,44.829


In [5]:
df_long = df.melt(
    id_vars=["COUNTRY", "INDICATOR"],
    var_name="YEAR",
    value_name="VALUE"
)


In [6]:
df_long["YEAR"] = pd.to_numeric(df_long["YEAR"], errors="coerce")
df_long = df_long.dropna(subset=["YEAR"])
df_long["YEAR"] = df_long["YEAR"].astype(int)

In [7]:
df_panel = df_long.pivot_table(
    index=["COUNTRY", "YEAR"],
    columns="INDICATOR",
    values="VALUE"
).reset_index()

In [8]:
df_panel.head()

INDICATOR,COUNTRY,YEAR,"All Items, Consumer price index (CPI), Period average, percent change","Current account balance (credit less debit), Percent of GDP","Expenditure, General government, Percent of GDP","Exports of goods and services, Volume, Free on board (FOB), Percent change","Gross capital formation, Percent of GDP","Gross debt, General government, Percent of GDP","Gross domestic product (GDP), Constant prices, Percent change","Gross national savings, Percent of GDP","Imports of goods and services, Volume, Cost insurance freight (CIF), Percent change","Net lending (+) / net borrowing (-), General government, Percent of GDP","Revenue, General government, Percent of GDP",Unemployment rate
0,"Afghanistan, Islamic Republic of",2002,NaN,33.908,6.943,NaN,27.243,345.977,NaN,61.151,NaN,-0.098,6.845,NaN
1,"Afghanistan, Islamic Republic of",2003,35.663,29.616,11.927,49.541,30.102,270.602,8.692,59.718,36.222,-2.102,9.826,NaN
2,"Afghanistan, Islamic Republic of",2004,16.358,37.216,15.069,-8.436,35.354,244.967,0.671,72.57,-0.427,-2.393,12.676,NaN
3,"Afghanistan, Islamic Republic of",2005,10.569,30.226,15.651,41.968,37.048,206.356,11.83,67.274,55.01,-0.917,14.733,NaN
4,"Afghanistan, Islamic Republic of",2006,6.785,20.844,18.262,-6.919,29.489,22.985,5.361,50.333,-2.198,0.684,18.946,NaN


In [9]:
df_panel.columns

Index(['COUNTRY', 'YEAR',
       'All Items, Consumer price index (CPI), Period average, percent change',
       'Current account balance (credit less debit), Percent of GDP',
       'Expenditure, General government, Percent of GDP',
       'Exports of goods and services, Volume, Free on board (FOB), Percent change',
       'Gross capital formation, Percent of GDP',
       'Gross debt, General government, Percent of GDP',
       'Gross domestic product (GDP), Constant prices, Percent change',
       'Gross national savings, Percent of GDP',
       'Imports of goods and services, Volume, Cost insurance freight (CIF), Percent change',
       'Net lending (+) / net borrowing (-), General government, Percent of GDP',
       'Revenue, General government, Percent of GDP', 'Unemployment rate'],
      dtype='object', name='INDICATOR')

Renaming columns

In [10]:
df_panel.rename(columns={
    "All Items, Consumer price index (CPI), Period average, percent change": "Inflation",
    "Gross domestic product (GDP), Constant prices, percent change": "GDP_Growth",
    "Unemployment rate": "Unemployment",
    "Gross debt, General government, Percent of GDP": "Debt",
    "Net lending (+) / net borrowing (-), General government, Percent of GDP": "Fiscal_Balance",
    "Current account balance (credit less debit), Percent of GDP": "Current_Account",
    "Exports of goods and services, volume, Free on board (FOB), Percent change": "Exports",
    "Imports of goods and services, volume, Cost insurance freight (CIF), Percent change": "Imports",
    "Revenue, General government, Percent of GDP": "Revenue",
    "Expenditure, General government, Percent of GDP": "Expenditure",
    "Gross national savings, Percent of GDP": "Savings",
    "Gross capital formation, Percent of GDP": "Investment"
}, inplace=True)

In [11]:
import matplotlib.pyplot as plt

country = "India"
var = "Inflation"

df_panel[df_panel["COUNTRY"] == country].plot(
    x="YEAR", y=var, title=f"{var} - {country}"
)
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
from statsmodels.tsa.stattools import adfuller

series = df_panel[df_panel["COUNTRY"]=="India"]["Inflation"].dropna()

result = adfuller(series)

print("ADF Statistic:", result[0])
print("p-value:", result[1])

ADF Statistic: -3.4752796769303402
p-value: 0.00864512983419235


In [ ]:
!pip install linearmodels

In [ ]:
from linearmodels.panel import unitroot

panel = df_panel.set_index(["COUNTRY","YEAR"])

unitroot.ImPesaranShin(panel["Inflation"])

ModuleNotFoundError: No module named 'linearmodels'